# **Load Libraries**

In [1]:
from unsloth import FastLanguageModel

from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
from sklearn.model_selection import train_test_split

from typing import List, Dict, Optional, Any
from tqdm.auto import tqdm

import re

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# **Load Dataset**

In [2]:
def load_and_merge_datasets(
    dataset_configs: List[Dict[str, Any]], 
    split: Optional[str] = None
) -> Dataset | DatasetDict:
    if not dataset_configs:
        raise ValueError("`dataset_configs` must contain at least one dataset.")
    
    loaded = []
    for config in tqdm(dataset_configs, desc='INFO: Loading Dataset'):
        config = config.copy()
        path = config.pop("path")
        effc_split = config.pop("split", split)

        try:
            ds = load_dataset(path, split=effc_split, **config)
            loaded.append(ds)

            print(f'Loaded dataset: {config} !')
        except:
            print(f'Error loading the dataset: {config}')

    
    if len(loaded) == 1:
        return loaded[0]
    
    if all(isinstance(ds, Dataset) for ds in loaded):
        return concatenate_datasets(loaded)

In [3]:
ds = load_and_merge_datasets(
    [
        {'path': 'arm-team/Stage1_SFT_aqua_rat', 'name': 'multiple_choice'},
        {'path': 'arm-team/Stage1_SFT_aqua_rat', 'name': 'open_form'}
    ],
    split='train'
)

INFO: Loading Dataset:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded dataset: {'name': 'multiple_choice'} !
Loaded dataset: {'name': 'open_form'} !


# **Find tags & pre-process**

In [4]:
TAG_PATTERNS: Dict[str, re.Pattern] = {
    'LONG_COT': re.compile(r'<LONG_COT>[\s\S]*?<\/LONG_COT>'),
    'COT':      re.compile(r'<COT>[\s\S]*?<\/COT>'),
    'CODE':     re.compile(r'<CODE>[\s\S]*?<\/CODE>'),
}
ANSWER_TAG: re.Pattern = re.compile(r'<ANSWER>[\s\S]*?<\/ANSWER>')

In [5]:
def assign_tag(example: Dict[str, str]) -> Dict[str, str]:
    tag_out: str = 'Direct'
    answer: str = example['output']

    for name, pattern in TAG_PATTERNS.items():
        if pattern.search(answer):
            tag_out = name
            break
    
    example['Tag'] = tag_out
    return example

In [6]:
# Assign tags based on output variables
ds_tagged = ds.map(assign_tag)

# Transform into pandas Dataframe format
df = ds_tagged.to_pandas()

In [7]:
print('Number of Tags used across dataset:')
print(df['Tag'].value_counts())

Number of Tags used across dataset:
Tag
Direct      10826
LONG_COT    10826
CODE        10826
COT         10826
Name: count, dtype: int64


# **Find token length & pre-process**

In [8]:
model_name = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 4096

In [9]:
# Load Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False
)

# Define & add special tokens to tokenizer
format_tag = [
    '<ANSWER>',   '</ANSWER>',
    '<CODE>',     '</CODE>',
    '<COT>',      '</COT>',
    '<LONG_COT>', '<LONG_COT>'
]
tokenizer.add_tokens(format_tag, special_tokens=True)

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce GTX 1650. Num GPUs = 1. Max memory: 3.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the cpu.


unsloth/Qwen3-4B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


7

In [10]:
def tokenize(example: Dict[str, str]) -> Dict[str, str | int]:
    instruction = example['instruction']
    solution = example['output']

    message = [
        {'role': 'user',      'content': instruction},
        {'role': 'assistant', 'content': solution   }
    ]

    msg_tokenized = tokenizer.apply_chat_template(
        message,
        tokenize=True,
        add_generation_prompt=False
    )['input_ids']

    example['Token_Length'] = len(msg_tokenized)
    return example

In [11]:
# Assign token length based on output variables
ds_full = ds_tagged.map(tokenize)

# Transform into pandas Dataframe format
df = ds_full.to_pandas()

In [12]:
# Create stratified train & test splits based on `Tag` column
df_train, df_test = train_test_split(df, test_size=0.1, stratify=df['Tag'])